# Phase 1 — COVID-19 Dataset Exploration

**Goal:** Understand the raw dataset before building any ETL pipeline.

**Dataset:** Our World in Data — COVID-19  
**Licence:** Creative Commons BY 4.0  
**URL:** https://covid.ourworldindata.org/data/owid-covid-data.csv

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('Libraries loaded OK')

## Step 1 — Load the dataset

In [ ]:
URL = 'https://covid.ourworldindata.org/data/owid-covid-data.csv'

# Download and save locally for offline use
df = pd.read_csv(URL)
df.to_csv('../data/owid-covid-data.csv', index=False)

print(f'Rows    : {len(df):,}')
print(f'Columns : {len(df.columns)}')

## Step 2 — Shape and data types

In [ ]:
df.info()

In [ ]:
df.head()

## Step 3 — Null value analysis

In [ ]:
null_summary = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_pct':   (df.isnull().sum() / len(df) * 100).round(1)
}).sort_values('null_pct', ascending=False)

print(null_summary[null_summary['null_count'] > 0].to_string())

## Step 4 — Countries and date range

In [ ]:
print(f"Unique locations : {df['location'].nunique()}")
print(f"Date range       : {df['date'].min()} → {df['date'].max()}")
print(f"Continents       : {sorted(df['continent'].dropna().unique())}")

## Step 5 — Quick visualisation: global new cases over time

In [ ]:
# Filter out OWID aggregate rows, group by date
daily = (
    df[~df['iso_code'].str.startswith('OWID_', na=True)]
    .groupby('date')['new_cases']
    .sum()
    .reset_index()
)
daily['date'] = pd.to_datetime(daily['date'])
daily['rolling_7d'] = daily['new_cases'].rolling(7).mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(daily['date'], daily['new_cases'], color='steelblue', alpha=0.4, label='Daily')
ax.plot(daily['date'], daily['rolling_7d'], color='navy', lw=1.5, label='7-day avg')
ax.set_title('Global new COVID-19 cases')
ax.set_xlabel('Date')
ax.set_ylabel('New cases')
ax.legend()
plt.tight_layout()
plt.show()

## Step 6 — Key columns for the ETL pipeline

Based on exploration, these are the columns we will keep for the warehouse:

| Column | Table | Notes |
|--------|-------|-------|
| iso_code | dim_location | Primary key source |
| continent | dim_location | |
| location | dim_location | Country name |
| population | dim_location | |
| date | dim_date | Parse to datetime |
| new_cases | fact_covid_daily | Fill nulls with 0 |
| new_deaths | fact_covid_daily | Fill nulls with 0 |
| new_vaccinations | fact_covid_daily | Fill nulls with 0 |
| total_cases | fact_covid_daily | |
| total_deaths | fact_covid_daily | |
| total_vaccinations | fact_covid_daily | |
| reproduction_rate | fact_covid_daily | Leave nulls as-is |

In [ ]:
# Summary statistics for key numeric columns
key_cols = ['new_cases','new_deaths','new_vaccinations','reproduction_rate']
df[key_cols].describe()